# 🏠 TP1 — Prédire le Loyer d'un Logement à Abidjan
## Module Data Science — Machine Learning (TP guidé)

> **Dataset :** `loyers_abidjan.csv` (6 000 logements) — à télécharger dans Colab

Problème de **RÉGRESSION** : prédire le loyer (un nombre) à partir des caractéristiques.
On compare les **3 algorithmes vus en cours** : régression linéaire, arbre de décision, KNN.

### Étapes
1. Exploration  2. Nettoyage  3. Préparation  4. Régression linéaire
5. Évaluation  6. Arbre & KNN  7. Interprétation & prédiction

In [ ]:
!pip install scikit-learn seaborn -q
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sns.set_theme(style="whitegrid")
print("Pret")

## 1. Exploration (EDA)

In [ ]:
df = pd.read_csv("loyers_abidjan.csv")
print("Dimensions :", df.shape)
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
print("Valeurs manquantes :\n", df.isna().sum())

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["loyer_fcfa"], bins=50, kde=True)
plt.title("Distribution des loyers"); plt.xlabel("Loyer (FCFA)")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
df.groupby("commune")["loyer_fcfa"].median().sort_values().plot(kind="barh")
plt.title("Loyer median par commune"); plt.xlabel("FCFA")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="superficie_m2", y="loyer_fcfa", alpha=0.3)
plt.title("Superficie vs Loyer")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlations")
plt.show()

## 2. Nettoyage

In [ ]:
for col in ["superficie_m2","age_batiment","nb_pieces"]:
    df[col] = df[col].fillna(df[col].median())
df = df.drop_duplicates().reset_index(drop=True)
print("Manquants restants :", df.isna().sum().sum())
print("Dimensions :", df.shape)

## 3. Preparation pour le ML

In [ ]:
df_ml = pd.get_dummies(df, columns=["commune","standing"], drop_first=True)
X = df_ml.drop(columns="loyer_fcfa")
y = df_ml["loyer_fcfa"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalisation (pour le KNN)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train : {len(X_train)} | Test : {len(X_test)}")

## 4. Modele 1 - Regression lineaire

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print("Regression lineaire entrainee")

## 5. Evaluation

In [ ]:
print("=== REGRESSION LINEAIRE ===")
print(f"MAE  : {mean_absolute_error(y_test, pred_lr):,.0f} FCFA")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_lr)):,.0f} FCFA")
print(f"R2   : {r2_score(y_test, pred_lr):.3f}")

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(y_test, pred_lr, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="Parfait")
plt.xlabel("Loyer reel"); plt.ylabel("Loyer predit")
plt.title("Predictions vs Realite"); plt.legend()
plt.show()

## 6. Modeles 2 & 3 - Arbre de decision & KNN

In [ ]:
# Arbre de decision (pas besoin de normalisation)
arbre = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train)
pred_arbre = arbre.predict(X_test)
print("=== ARBRE DE DECISION ===")
print(f"MAE : {mean_absolute_error(y_test, pred_arbre):,.0f} FCFA | R2 : {r2_score(y_test, pred_arbre):.3f}")

In [ ]:
# ATTENTION a l'overfitting : tester la profondeur
print("Effet de la profondeur de l'arbre :")
for prof in [3, 5, 8, 12, None]:
    a = DecisionTreeRegressor(max_depth=prof, random_state=42).fit(X_train, y_train)
    print(f"  max_depth={str(prof):5} -> R2 = {r2_score(y_test, a.predict(X_test)):.3f}")
# trop profond (None) n'est PAS le meilleur : c'est l'overfitting !

In [ ]:
# KNN (NORMALISATION obligatoire)
knn = KNeighborsRegressor(n_neighbors=5).fit(X_train_s, y_train)
pred_knn = knn.predict(X_test_s)
print("=== KNN (k=5) ===")
print(f"MAE : {mean_absolute_error(y_test, pred_knn):,.0f} FCFA | R2 : {r2_score(y_test, pred_knn):.3f}")

In [ ]:
# Comparaison des 3 modeles vus en cours
comp = pd.DataFrame({
    "Modele": ["Regression lineaire", "Arbre de decision", "KNN (k=5)"],
    "R2": [r2_score(y_test, pred_lr), r2_score(y_test, pred_arbre), r2_score(y_test, pred_knn)],
    "MAE": [mean_absolute_error(y_test, pred_lr), mean_absolute_error(y_test, pred_arbre),
            mean_absolute_error(y_test, pred_knn)]
})
print(comp)
print("\n-> Le KNN gagne ici : pas de meilleur algorithme universel, on teste et on compare !")

## 7. Interpretation & prediction

In [ ]:
# Importance des variables (via l'arbre de decision)
importances = pd.Series(arbre.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(10,6))
importances.plot(kind="barh")
plt.title("Top 10 variables importantes"); plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Prediction avec le KNN (meilleur modele) - ne pas oublier de normaliser !
def predire_loyer_knn(modele, scaler, colonnes, **carac):
    logement = pd.DataFrame(0, index=[0], columns=colonnes)
    for k, v in carac.items():
        if k in logement.columns:
            logement[k] = v
    return modele.predict(scaler.transform(logement))[0]

loyer = predire_loyer_knn(knn, scaler, X.columns,
    superficie_m2=80, nb_pieces=3, age_batiment=5, etage=2,
    parking=1, securite=1, meuble=0, commune_Cocody=1, standing_Moyen=1)
print(f"Loyer estime : {loyer:,.0f} FCFA/mois")

## Pour aller plus loin

In [ ]:
# Trouver le meilleur K
scores_k = {}
for k in range(1, 21):
    knn_k = KNeighborsRegressor(n_neighbors=k).fit(X_train_s, y_train)
    scores_k[k] = r2_score(y_test, knn_k.predict(X_test_s))
bk = max(scores_k, key=scores_k.get)
print(f"Meilleur K : {bk} (R2 = {scores_k[bk]:.3f})")

# Validation croisee
scores = cross_val_score(lr, X, y, cv=5, scoring="r2")
print(f"R2 moyen regression lineaire (5-fold) : {scores.mean():.3f} +/- {scores.std():.3f}")

---
## TP1 termine !

Vous avez compare les 3 algorithmes vus en cours (regression lineaire, arbre, KNN),
identifie les facteurs du loyer, et cree une fonction de prediction.

*Module Data Science — TP1 : Prediction de loyer | Bootcamp Data Science*